In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [12]:
# 定义数据集类
class ShapeDataset(Dataset):
    def __init__(self, num_samples=1000):
        self.shapes = torch.randn(num_samples, 2)  # 随机生成形状的参数
        self.areas = self.shapes[:, 0] * self.shapes[:, 1]  # 计算面积

    def __len__(self):
        return len(self.shapes)

    def __getitem__(self, idx):
        return self.shapes[idx], self.areas[idx]

In [13]:
# 定义模型
class AreaModel(nn.Module):
    def __init__(self):
        super(AreaModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.model(x)

In [14]:
# 训练模型
def train_model(model, criterion, optimizer, dataloader, device, epochs=200):
    model.to(device)  # 将模型移动到 GPU
    for epoch in range(epochs):
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.unsqueeze(1).to(device)  # 将数据移动到 GPU
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 100 == 0:  # 每100轮打印一次损失
            print(f'Epoch {epoch+1}, Loss: {loss.item()}')

In [15]:

# 主程序
if __name__ == "__main__":
    # 检查 CUDA 是否可用
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Using device: {device}')

    # 生成数据
    num_samples = 1000
    dataset = ShapeDataset(num_samples)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    # 初始化模型
    model = AreaModel()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # 训练模型
    train_model(model, criterion, optimizer, dataloader, device, epochs=800)


Using device: cuda
Epoch 100, Loss: 8.606536721345037e-05
Epoch 200, Loss: 9.789464820642024e-05
Epoch 300, Loss: 0.0009765181457623839
Epoch 400, Loss: 0.010259456932544708
Epoch 500, Loss: 0.00021701885270886123
Epoch 600, Loss: 0.0011762267677113414
Epoch 700, Loss: 4.0667720895726234e-05
Epoch 800, Loss: 0.004048981238156557
